# Importing Libraries & Dependencies

In [ ]:
import nltk
import pandas as pd
import spacy


In [ ]:
# This only needs to be run once to download the model

# Uncomment to use the large model
# spacy.cli.download("en_core_web_lg")
# nlp = spacy.load("en_core_web_lg")

# Using the small model
spacy.cli.download("en_core_web_sm")
nlp = spacy.load("en_core_web_sm")

nltk.download('wordnet')

# Loading the Dataset

In [ ]:
df = pd.read_csv('data.csv').dropna()
df.head()

In [ ]:
# Inspecting the dataset

df['Department'].value_counts()

In [ ]:
df['Priority'].value_counts()

# Basic Preprocessing (Regex)

In [ ]:
import re
# Basic text preprocessing function using regex
def basic_preprocess(text):
    
    # lowercase
    text = text.lower()
    
    # remove punctuation and special characters
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    # remove digits
    text = re.sub(r'\d+', ' ', text)

    return text


In [ ]:
text = df.iloc[10]['Body']
text

In [ ]:
# Basic Preprocessing
basic_preprocess(text)

# Stemming

In [ ]:
# Stemming example

from nltk.stem import PorterStemmer, LancasterStemmer
stemmer = PorterStemmer()

stemmed_text = " ".join([stemmer.stem(word) for word in basic_preprocess(text).split()])
stemmed_text

# Lemmatize

In [ ]:
# Lemmatization example
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

lemmatized_text = " ".join([lemmatizer.lemmatize(word, pos='v') for word in text.split()])
lemmatized_text

# Named Entity Recognition

In [ ]:
# Named Entity Recognition (NER)

doc = nlp(text)
for ent in doc.ents:
    print(ent.text, ent.label_)

# Part-of-Speech (POS) Taggin

In [ ]:
# Part-of-Speech (POS) Tagging
doc = nlp(text)
for token in doc:
    print(f"{token.text:20} {token.pos_:10} {token.tag_}")

# Tokenization & Vectorization

In [ ]:
# Vectorization using CountVectorizer and TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

count_vectorizer = CountVectorizer(max_features=5, stop_words='english', ngram_range=(1,1)) # Showing 5 features just as sample
tfidf_vectorizer = TfidfVectorizer(max_features=5, stop_words='english', ngram_range=(1,1)) # Showing 5 features just as sample

count_vectors = count_vectorizer.fit_transform(df['Body'])
tfidf_vectors = tfidf_vectorizer.fit_transform(df['Body'])



In [ ]:
# Display vectors in table
count_df = pd.DataFrame(count_vectors.toarray(), columns=count_vectorizer.get_feature_names_out())
tfidf_df = pd.DataFrame(tfidf_vectors.toarray(), columns=tfidf_vectorizer.get_feature_names_out())

# Combine with department column
count_df = pd.concat([count_df, df['Priority'].reset_index(drop=True)], axis=1)
tfidf_df = pd.concat([tfidf_df, df['Priority'].reset_index(drop=True)], axis=1)

count_df

# Putting it all together

In [ ]:
lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()

def preprocess_text(text):
    # lowercase
    text = text.lower()
    
    # remove punctuation and special characters
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    # remove digits
    text = re.sub(r'\d+', ' ', text)
    
    # lemmatization
    text = " ".join([lemmatizer.lemmatize(word, pos='v') for word in text.split()])

    # stemming
    text = " ".join([stemmer.stem(word) for word in text.split()])

    # masking named entities - takes longer to process.
    # Uncomment to implement 
    # doc = nlp(text)
    # for ent in doc.ents:
    #     text = text.replace(ent.text, ent.label_)

    return text

In [ ]:
df['preprocessed_Body'] = df['Body'].apply(preprocess_text)
df

# Model Training

In [ ]:
# Train-Test Split

from sklearn.model_selection import train_test_split

X = df['preprocessed_Body'] # Features
y = df['Priority'] # Target variable

X_train, X_test, y_train, y_test = train_test_split(
    X, # X
    y, # Y
    test_size=0.2,
    random_state=42, 
    stratify=y
    )

### Using Bayesian classifier (1,1) ngram

In [ ]:
# naive bayes classifier
from sklearn import pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline


# creating the model with pipeline
model_nb = Pipeline([
    ('vectorizer', TfidfVectorizer(
        stop_words='english',
        ngram_range=(1,1),
        max_df = 0.85
    )),
    ('classifier', MultinomialNB())
])


model_nb.fit(X_train, y_train)
y_pred = model_nb.predict(X_test)

print(classification_report(y_test, y_pred))

### Bayesian Classifier (2,3) ngram

In [ ]:
# naive bayes classifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline

# creating the model with pipeline
model_nb = Pipeline([
    ('vectorizer', TfidfVectorizer(
        stop_words='english',
        ngram_range=(2,3),
        max_df = 0.85
    )),
    ('classifier', MultinomialNB())
])


model_nb.fit(X_train, y_train)
y_pred = model_nb.predict(X_test)

print(classification_report(y_test, y_pred))

### Logistic Regression Classifier

In [ ]:
# train logistic regression model
from sklearn.linear_model import LogisticRegression

model_lr = Pipeline([
    ('vectorizer', TfidfVectorizer(
        stop_words='english',
        ngram_range=(2,3),
        max_df = 0.85)
        
    ),
    ('classifier', LogisticRegression(max_iter=200))
])

model_lr.fit(X_train, y_train)
y_pred_lr = model_lr.predict(X_test)

print(classification_report(y_test, y_pred_lr))

### Random Forest Model

In [ ]:
# train random forest model
from sklearn.ensemble import RandomForestClassifier

model_rf = Pipeline([
    ('vectorizer', TfidfVectorizer(
        stop_words='english',
        ngram_range=(2,3),
        max_df = 0.85)
        
    ),
    ('classifier', RandomForestClassifier(n_estimators=100))
])
model_rf.fit(X_train, y_train)
y_pred_rf = model_rf.predict(X_test)

print(classification_report(y_test, y_pred_rf))

### XGBoost Classifier

In [ ]:
# train model using xgboost
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

# Encode target labels
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

model_xgb = Pipeline([
    ('vectorizer', TfidfVectorizer(
        stop_words='english',
        ngram_range=(2,3),
        max_df = 0.85)
        
    ),
    ('classifier', XGBClassifier(use_label_encoder=False, eval_metric='mlogloss'))
])
model_xgb.fit(X_train, y_train_encoded)
y_pred_xgb = model_xgb.predict(X_test)

print(classification_report(y_test_encoded, y_pred_xgb, target_names=label_encoder.classes_))

# Extras

In [ ]:
# Combining nlp with regression

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

df = pd.read_csv("data_v2.csv")
df['preprocessed_ticket_description'] = df['Ticket Description'].apply(preprocess_text)
df

In [ ]:
import seaborn as sns

sns.displot(df['Duration (hr)'])

In [ ]:
# Train-Test Split

from sklearn.model_selection import train_test_split

X = df['preprocessed_ticket_description'] # Features
y = df['Duration (hr)'] # Target variable

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    )

In [ ]:
# train polynomial regression
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures

model_linear_reg = Pipeline([
    ('vectorizer', TfidfVectorizer(
        stop_words='english',
        ngram_range=(2,3),
        max_df = 0.85,
        max_features = 100
        )
    ),
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('regressor', LinearRegression())
])

model_linear_reg.fit(X_train, y_train)
y_pred_linear_reg = model_linear_reg.predict(X_test)


In [ ]:
# Evaluate the model
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
print(f"Mean Absolute Error: {mean_absolute_error(y_test, y_pred_linear_reg):.4f}")
print(f"Root Mean Squared Error: {root_mean_squared_error(y_test, y_pred_linear_reg):.4f}")

In [ ]:
text = """
The product_purchased is not turning on. It was working fine until yesterday, but now it doesn't respond.
\n\n1.8.3 I really I'm using the original charger that came with my product_purchased, but it's not charging properly."
"""

input_text = preprocess_text(text)
predicted_duration = model_linear_reg.predict([input_text])
print(f"Predicted Duration (hr) for input text: {predicted_duration[0]:.2f} hours")